# 🚀 Master Persistent Google Colab Runner

### Why this runner is foolproof:
1. **Never redownload datasets or weights**: Everything heavy (CityFlow dataset, FastReID model weights, and computed embedding caches) is saved permanently in your **Google Drive** under `/MyDrive/trafficsurveillance_storage/`.
2. **Always up-to-date with GitHub**: Running Cell 2 automatically pulls the latest changes pushed from the IDE.
3. **Persistent results**: All experiment outputs and caches are saved directly to Google Drive so nothing is lost when Colab disconnects.

## Step 1: Mount Google Drive (Persistent Storage Anchor)

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Set up permanent directory in Drive
STORAGE_DIR = Path('/content/drive/MyDrive/trafficsurveillance_storage')
STORAGE_DIR.mkdir(parents=True, exist_ok=True)
(STORAGE_DIR / "weights").mkdir(exist_ok=True)
(STORAGE_DIR / "cache").mkdir(exist_ok=True)
(STORAGE_DIR / "outputs").mkdir(exist_ok=True)

print("✅ Permanent Google Drive storage ready at:", STORAGE_DIR)

## Step 2: Clone or Pull Latest Code from GitHub & Install

In [ ]:
%cd /content
!if [ -d "Vehicle-Tracking" ]; then cd Vehicle-Tracking && git pull origin main; else git clone https://github.com/neoaltair/Vehicle-Tracking.git; fi
%cd /content/Vehicle-Tracking

# Install project and dependencies (fast)
!pip install -q -e ".[dev]"
!pip install -q fastreid==1.4.0 --no-deps

import torch
print("\n✅ Environment ready. CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))

## Step 3: Link Persistent Data & Checkpoints (One-time Setup, Auto-Skipped Afterward)

In [ ]:
import glob

# 1. Model checkpoint in Drive
weights_drive = STORAGE_DIR / "weights" / "veri_sbs_R50-ibn.pth"
if not weights_drive.exists():
    print("Downloading VeRi-776 weights to Google Drive (one-time only)...\n")
    !wget -q -O "$weights_drive" https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth
    print("✅ Downloaded weights to Drive.")
else:
    print("✅ Found existing model weights in Drive:", weights_drive)

# 2. Dataset in Drive
dataset_drive = STORAGE_DIR / "cityflow"
gt_files = glob.glob(str(dataset_drive / "**/gt.txt"), recursive=True)

if not gt_files:
    print("Extracting CityFlow to Google Drive (one-time only, this takes ~2-3 min)...\n")
    !pip install -q gdown
    !gdown 13wNJpS_Oaoe-7y5Dzexg_Ol7bKu1OWuC -O /content/dataset_temp.zip
    dataset_drive.mkdir(parents=True, exist_ok=True)
    !unzip -q /content/dataset_temp.zip -d "$dataset_drive"
    !rm -f /content/dataset_temp.zip
    gt_files = glob.glob(str(dataset_drive / "**/gt.txt"), recursive=True)
    print(f"✅ Extracted CityFlow to Drive! Found {len(gt_files)} cameras.")
else:
    print(f"✅ Found existing CityFlow dataset in Drive ({len(gt_files)} cameras) at: {dataset_drive}")

# 3. Symlink into repo outputs/
!mkdir -p outputs/weights
!ln -sfn "$weights_drive" outputs/weights/veri_sbs_R50-ibn.pth
!ln -sfn "$STORAGE_DIR/cache" outputs/cache
print("✅ Linked Drive storage to repo outputs/")

## Step 4: Ensure Gallery Embeddings Exist in Drive Cache

If the embedding cache is already in your Google Drive, this step completes in ~1 second (skips existing files). If not, it extracts and caches them directly into your Google Drive permanently.

In [ ]:
# Find root directory that contains 'train'
root_candidates = [dataset_drive] + [Path(p).parent.parent.parent for p in gt_files if 'train' in p]
actual_root = str(root_candidates[0])
for r in root_candidates:
    if (Path(r) / "train").exists():
        actual_root = str(r)
        break

print("Using CityFlow root:", actual_root)

!python scripts/extract_m3_embeddings.py \
  --root "$actual_root" \
  --crops-dir data/processed/cityflowv2/crops \
  --weights outputs/weights/veri_sbs_R50-ibn.pth \
  --embedding-cache outputs/cache/embeddings

## Step 5: Run the M4 Corrected Degradation Study

Executes the controlled query degradation evaluation (Downscale, Blur, Crop, Occlusion) comparing Appearance-only vs. Spatio-Temporal fusion.

In [ ]:
!python scripts/run_m4_degradation_study.py \
  --root "$actual_root" \
  --embedding-cache outputs/cache/embeddings \
  --weights outputs/weights/veri_sbs_R50-ibn.pth \
  --time-window-s 60.0 \
  --alpha 0.05 \
  --out-dir outputs/m4_degradation

# Copy results to permanent Drive storage as backup
!cp -r outputs/m4_degradation "$STORAGE_DIR/outputs/"
print("\n✅ Results backed up permanently to Google Drive at:", STORAGE_DIR / "outputs/m4_degradation")

## Step 6: Commit and Push Results Back to GitHub

If you want to push the newly generated `degradation_results.json` directly from Colab:

In [ ]:
# View degradation results table
import json
results_path = Path("outputs/m4_degradation/degradation_results.json")
if results_path.exists():
    data = json.loads(results_path.read_text())
    print(f"Degradation results generated for {len(data)} conditions:")
    for row in data:
        print(f"{row['family']:<10} sev={row['severity']:<5} | App mAP={row['app_mAP_mean']:.4f} -> ST mAP={row['st_mAP_mean']:.4f} (gain={row['mAP_gain']:+.4f})")
else:
    print("Results file not found.")